In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import Imagenette, SVHN
import numpy as np

import logging

import importlib
import FL_sim
import models_to_train
import compressor.entropy_coding, quantizer.simple, quantizer.wz_quant_ANN

importlib.reload(compressor.entropy_coding)
importlib.reload(quantizer.simple)
importlib.reload(quantizer.wz_quant_ANN)
importlib.reload(FL_sim)
importlib.reload(models_to_train)

from models_to_train import ResNetPLModel
from FL_sim import FLSimulator, save_grads_f_applied_on_grads
from compressor.entropy_coding import entropy_coding, entropy_decoding
from typing import Dict, List
from quantizer.simple import simple_quantize, simple_dequantize
from quantizer.wz_quant_ANN import WZQuantizer

torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

In [3]:
# def test_model_training():
#     from pytorch_lightning import Trainer
#     model = ResNetPLModel(num_classes=10,
#         resnet_version='resnet18', lr=0.005)
#     training_dataloader = torch.utils.data.DataLoader(
#         dataset[0], batch_size=14000, shuffle=True,
#         num_workers=10, persistent_workers=True)
#     test_dataloader = torch.utils.data.DataLoader(
#         dataset[1], batch_size=14000,
#         num_workers=2, persistent_workers=True)
#     trainer = Trainer(max_epochs=10, accelerator='cuda', log_every_n_steps=len(training_dataloader)//2)
#     trainer.fit(model, training_dataloader, test_dataloader)
# # test_model_training()

In [4]:
# dataset = [torch.utils.data.Subset(d, list(range(50))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

In [5]:



def dict_to_array_and_normalize(grad_dict: Dict, min_v: List, max_v: List):
    res = []
    for i, (k, v) in enumerate(grad_dict.items()):
        v = v.ravel() * 1000
        v = (v - min_v[i] * 1000) / (max_v[i] * 1000 - min_v[i] * 1000)
        res.append(v.to('cpu').numpy())
    res = np.concatenate(res)
    return res


def recover_shape_and_denormal_to_dict(grad_vector, org_shapes_dict, min_v: List, max_v: List):
    res = {}
    start = 0
    for i, (k, shape) in enumerate(org_shapes_dict.items()):
        end = start + np.prod(shape)
        v = grad_vector[start:end]
        v = v * (max_v[i] - min_v[i]) + min_v[i]
        res[k] = v.reshape(shape)
        start = end
    return res


wz_quantizer = WZQuantizer()

In [6]:
keys_state = [k for k, v in ResNetPLModel(
    num_classes=10, resnet_version='resnet18'
).named_parameters() if v.requires_grad]
r_original_dict_list = None
r_byte_size_dict_list = None
r_decomp_error_dict_list = None
r_total_error = None
state_report_per_round = []


# make a wrapper to report compression information
def report_compression_stat(func):
    def wrapper(worker_grad_dict, agent_id):
        global r_original_dict_list, r_byte_size_dict_list, \
            r_decomp_error_dict_list, r_total_error, keys_state

        if r_total_error is None or (agent_id == 0 and len(r_total_error) > 1):
            state_report_per_round.append({
                'byte size info per agent': r_byte_size_dict_list,
                '% error per layer per agent': r_decomp_error_dict_list,
                '% total error per agent': r_total_error,
            })
            r_original_dict_list = {k: [] for k in keys_state}
            r_byte_size_dict_list = {k: [] for k in ['raw', 'wz', 'entropy']}
            r_decomp_error_dict_list = {k: [] for k in keys_state}
            r_total_error = []

        assert agent_id == len(r_original_dict_list[keys_state[0]]), \
            "something wrong with the order of execution, agent_id given too soon"

        r_original_dict_list = {k: v + [worker_grad_dict[k]]
                                for k, v in r_original_dict_list.items()}

        encoded_data, min_v, max_v, dtype = func(worker_grad_dict, agent_id)

        # simulate entropy only coding
        entr_grad_flat_normal = dict_to_array_and_normalize(worker_grad_dict, min_v, max_v)
        entr_quantized_data = simple_quantize(entr_grad_flat_normal)
        entr_encoded_data = entropy_coding(entr_quantized_data)

        enc_data_byte_size = len(encoded_data)
        org_data_byte_size = sum(v[0].nbytes for v in worker_grad_dict.values())
        entr_enc_data_byte_size = len(entr_encoded_data)

        r_byte_size_dict_list['raw'].append(org_data_byte_size)
        r_byte_size_dict_list['wz'].append(enc_data_byte_size)
        r_byte_size_dict_list['entropy'].append(entr_enc_data_byte_size)

        return encoded_data, min_v, max_v, dtype

    return wrapper


def report_decompression_stat(func):
    def wrapper(agent_id, global_model_dims, previous_data, worker_broadcast_data):
        global r_original_dict_list, r_decomp_error_dict_list, r_total_error, keys_state
        assert agent_id == len(r_decomp_error_dict_list[keys_state[0]]), \
            "something wrong with the order of execution, agent_id given too soon"

        result_dict = func(agent_id, global_model_dims, previous_data, worker_broadcast_data)

        error_dict = {}
        total_error = 0
        total_element_count = sum(v[0].numel() for v in r_original_dict_list.values())
        for k, v in result_dict.items():
            v = torch.Tensor(v).cuda()
            temp = r_original_dict_list[k][agent_id]
            error_dict[k] = (v - temp).abs().sum()
            error_dict[k] = (error_dict[k] / temp.abs().sum()).item()
            total_error += error_dict[k] * (temp.numel() / total_element_count)

        r_decomp_error_dict_list = {k: v + [error_dict[k]]
                                    for k, v in r_decomp_error_dict_list.items()}
        r_total_error.append(total_error)

        return result_dict

    return wrapper


In [7]:
@report_compression_stat
def pre_send_process(worker_grad_dict, agent_id):
    min_v, max_v = [[f(v).to('cpu').numpy()
                     for k, v in worker_grad_dict.items()] for f in [torch.min, torch.max]]

    # worker_grad_dict={k:v*1.1 for k, v in worker_grad_dict.items()}
    # return worker_grad_dict, min_v, max_v, 0

    grad_flat_normal = dict_to_array_and_normalize(worker_grad_dict, min_v, max_v)

    quantizer = simple_quantize if agent_id >= 1 else wz_quantizer.encode
    quantized_data = quantizer(grad_flat_normal)

    dtype = quantized_data.dtype
    encoded_data = entropy_coding(quantized_data)

    return encoded_data, min_v, max_v, dtype


@report_decompression_stat
def server_rec_process(agent_id, global_model_dims, previous_data, worker_broadcast_data):
    # return worker_broadcast_data[0]
    encoded_data, min_v, max_v, dtype = worker_broadcast_data

    quantized_decoded_data = entropy_decoding(encoded_data, dtype)

    if agent_id >= 1:
        res_vector = simple_dequantize(quantized_decoded_data, np.float32)
        if agent_id == 0:
            wz_quantizer.initialize_x_info_dataset(res_vector)
        elif agent_id == 1:
            wz_quantizer.train_model(res_vector)
    else:
        res_vector = wz_quantizer.decode(quantized_decoded_data, previous_data)

    result_dict = recover_shape_and_denormal_to_dict(
        res_vector, global_model_dims, min_v, max_v)

    result_dict = {k: torch.tensor(v).to('cuda') for k, v in result_dict.items()}

    return result_dict


def applied_on_grads_before_optim(
        fl_model, worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs):
    pass

In [ ]:
from experiments.resnet_parameter_corr_between_worker import load_grad_files

load_grad_files([[0,0], [0, 1]], keys_state, path_to_files, curr_round, current_epoch=1, single_worker=True)

In [8]:
k = 5  # Number of workers

batch_size = 13000
# batch_size /= 50 # imagenet
# batch_size /= 6 # more batches as samples
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)

# applied_on_grads_before_optim = save_grads_f_applied_on_grads
# model.args_for_f_on_grad['save_folder_path'] = \
#     f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01,
                      logging_disabled=False, applied_on_grads_before_optim=applied_on_grads_before_optim)

model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))

sim = FLSimulator(
    num_agents=k, communication_rounds=10, client_epochs_per_round=15,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_flag=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

# pre-training global model before saving it
# sim.do_train_global_model_and_set_local_model(num_epochs=4)
# torch.save(sim.global_model.state_dict(), 'experiments/exp_data/resnet18_svhn.pth')

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
sim.run_simulation(metric_logging=True, pre_training_global_epochs=0)


round 1/10 --------------------
  - reporting global model metrics
         test loss: 2.168, test auc: 0.614
         train loss: (rank=0) 2.177, train auc: 0.614
     > training agent 1/3
         test loss: 2.019, test auc: 0.702
         train loss: (rank=0) 

Missing logger folder: experiments/exp_data/run_stats\agent_1_round_0


1.972, train auc: 0.726
     > training agent 2/3
         test loss: 1.988, test auc: 0.715
         train loss: (rank=1) 

Missing logger folder: experiments/exp_data/run_stats\agent_2_round_0


1.957, train auc: 0.737
     > training agent 3/3
         test loss: 1.947, test auc: 0.739
         train loss: (rank=2) 1.923, train auc: 0.758
Aggregating gradients using FedAvg...

round 2/10 --------------------
  - reporting global model metrics
         test loss: 2.084, test auc: 0.687
         train loss: (rank=0) 

...

1.493, train auc: 0.893
     > training agent 1/3
         test loss: 0.922, test auc: 0.949
         train loss: (rank=0) 

Missing logger folder: experiments/exp_data/run_stats\agent_1_round_9


0.813, train auc: 0.962
     > training agent 2/3
         test loss: 0.963, test auc: 0.945
         train loss: (rank=1) 

Missing logger folder: experiments/exp_data/run_stats\agent_2_round_9


0.874, train auc: 0.955
     > training agent 3/3
         test loss: 0.921, test auc: 0.949
         train loss: (rank=2) 0.822, train auc: 0.961
Aggregating gradients using FedAvg...

final global model metrics
         test loss: 1.385, test auc: 0.909
         train loss: (rank=1) 1.455, train auc: 0.899


In [9]:
torch.save(
    state_report_per_round,
    f'experiments/exp_data/compression_stats_report/compression_stats.pth',
)